In [13]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import math
import openpyxl

In [2]:
player_info_df = pd.read_csv("player_info.csv", encoding="utf-8", sep = ",", header=0)
player_info_df.head()


,PLAYER,POS,TEAM,PPG
0,TUCKER,K,BAL,8.0
1,JACKSON,QB,BAL,23.1
2,HENRY,RB,BAL,30.6
3,LIKELY,TE,BAL,6.8
4,ALLEN,QB,BUF,23.0


In [16]:
rosters_list_df = pd.read_csv("rosters_list.csv", encoding="utf-8", sep = ",", header=0).iloc[:,0:8]
rosters_list_df.head()

,MANAGER,PLAYER,POS,WC_PTS,DIV_PTS,CONF_PTS,SB_PTS,TOTAL_PTS
0,TEAM RAHEEM,ALLEN,QB,0,0,0,0,0
1,TEAM RAHEEM,JACKSON,QB,0,0,0,0,0
2,TEAM RAHEEM,GIBBS,RB,0,0,0,0,0
3,TEAM RAHEEM,JACOBS,RB,0,0,0,0,0
4,TEAM RAHEEM,BARKLEY,RB,0,0,0,0,0


In [5]:
team_odds_df = pd.read_csv("team_odds.csv", encoding="utf-8", sep = ",", header=0)
team_odds_df.head()

,CONF,SEED,TEAM,WC_WP,DIV_AP,CONF_AP,REMAINING,DET,PHI,LAR,...,MIN,WAS,GB,KC,BUF,BAL,HOU,PIT,LAC,DEN
0,AFC,1,DET,1.00,0.6688,0.361963,2.030763,NaN,0.50,0.55,...,0.65,0.7,0.70,0.50,0.50,0.55,0.60,0.65,0.70,0.70
1,AFC,2,PHI,0.65,0.3757,0.220292,2.245992,0.50,NaN,0.55,...,0.65,0.7,0.65,0.50,0.50,0.55,0.60,0.65,0.70,0.70
2,AFC,3,LAR,0.60,0.2952,0.155875,2.051075,0.45,0.45,NaN,...,0.60,0.6,0.70,0.40,0.45,0.50,0.55,0.60,0.70,0.70
3,AFC,4,TB,0.60,0.2631,0.120275,1.983375,0.40,0.40,0.45,...,0.60,0.6,0.60,0.40,0.40,0.45,0.50,0.50,0.55,0.55
4,AFC,5,MIN,0.40,0.1526,0.060477,1.613077,0.35,0.35,0.40,...,NaN,0.5,0.50,0.35,0.40,0.45,0.50,0.50,0.50,0.50


In [38]:
team_odds_short = team_odds_df[["TEAM", "REMAINING"]]

rosters_pts_df = rosters_list_df.merge(player_info_df, on = ["POS", "PLAYER"], how="left")\
    .merge(team_odds_short, on = "TEAM", how = "left")
rosters_pts_df["FUT_PTS"] =  rosters_pts_df["PPG"] * rosters_pts_df["REMAINING"]
rosters_pts_df["PROJ_PTS"] = np.round(rosters_pts_df["TOTAL_PTS"] + rosters_pts_df["FUT_PTS"], 2)
rosters_pts_df["STILL_ALIVE"] = np.where(rosters_pts_df["REMAINING"]>0,1,0)
rosters_pts_df.head()

,MANAGER,PLAYER,POS,WC_PTS,DIV_PTS,CONF_PTS,SB_PTS,TOTAL_PTS,TEAM,PPG,REMAINING,FUT_PTS,PROJ_PTS,STILL_ALIVE
0,TEAM RAHEEM,ALLEN,QB,0,0,0,0,0,BUF,23.0,2.315335,53.252705,53.25,1
1,TEAM RAHEEM,JACKSON,QB,0,0,0,0,0,BAL,23.1,2.026491,46.811935,46.81,1
2,TEAM RAHEEM,GIBBS,RB,0,0,0,0,0,DET,18.0,2.030763,36.553733,36.55,1
3,TEAM RAHEEM,JACOBS,RB,0,0,0,0,0,GB,16.0,1.490690,23.851032,23.85,1
4,TEAM RAHEEM,BARKLEY,RB,0,0,0,0,0,PHI,20.0,2.245992,44.919834,44.92,1


In [39]:
standings_df = rosters_pts_df.groupby("MANAGER", as_index=False)\
    .agg({
        "TOTAL_PTS":"sum",
        "PROJ_PTS":"sum"
    })\
    .rename(columns = {
        "MANAGER": "Fantasy Team",
        "TOTAL_PTS": "Points",
        "PROJ_PTS": "Projected"
    })

standings_df.head()

,Fantasy Team,Points,Projected
0,ARNIE,0,374.25
1,BAKE SHOW,0,415.98
2,BILL ME LATER,0,443.85
3,JA4MVP,0,449.56
4,JACOBS LADDER,0,401.19


In [43]:
roster_view_df = rosters_pts_df[["MANAGER", "PLAYER", "POS", "TEAM", "TOTAL_PTS", "PROJ_PTS", "STILL_ALIVE", "PPG", "REMAINING", "WC_PTS", "DIV_PTS", "CONF_PTS", "SB_PTS"]]
roster_view_df["REMAINING"] = np.round(roster_view_df["REMAINING"], 2)
roster_view_df = roster_view_df.rename(columns={
        "MANAGER": "Fantasy Team",
        "PLAYER": "Player",
        "TEAM": "TM",
        "TOTAL_PTS": "Points",
        "PROJ_PTS": "Projected Points",
        "STILL_ALIVE": "Still Alive",
        "PPG": "Reg Szn PPG",
        "REMAINING": "Exp Games Remaining",
        "WC_PTS": "WC PTS",
        "DIV_PTS": "DIV PTS",
        "CONF_PTS": "CONF PTS",
        "SB_PTS": "SB PTS"
    })

roster_view_df.head()

C:\Users\Owner\AppData\Local\Temp\ipykernel_21508\2111211850.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  roster_view_df["REMAINING"] = np.round(roster_view_df["REMAINING"], 2)


,Fantasy Team,Player,POS,TM,Points,Projected Points,Still Alive,Reg Szn PPG,Exp Games Remaining,WC PTS,DIV PTS,CONF PTS,SB PTS
0,TEAM RAHEEM,ALLEN,QB,BUF,0,53.25,1,23.0,2.32,0,0,0,0
1,TEAM RAHEEM,JACKSON,QB,BAL,0,46.81,1,23.1,2.03,0,0,0,0
2,TEAM RAHEEM,GIBBS,RB,DET,0,36.55,1,18.0,2.03,0,0,0,0
3,TEAM RAHEEM,JACOBS,RB,GB,0,23.85,1,16.0,1.49,0,0,0,0
4,TEAM RAHEEM,BARKLEY,RB,PHI,0,44.92,1,20.0,2.25,0,0,0,0


In [47]:
baseline_roster_df = roster_view_df.loc[roster_view_df["Fantasy Team"]=="TEAM RAHEEM"]
baseline_roster_df = baseline_roster_df.drop(["Fantasy Team", "Reg Szn PPG", "Exp Games Remaining", "WC PTS", "DIV PTS", "CONF PTS", "SB PTS"], axis=1)

baseline_roster_df.head()

,Player,POS,TM,Points,Projected Points,Still Alive
0,ALLEN,QB,BUF,0,53.25,1
1,JACKSON,QB,BAL,0,46.81,1
2,GIBBS,RB,DET,0,36.55,1
3,JACOBS,RB,GB,0,23.85,1
4,BARKLEY,RB,PHI,0,44.92,1


In [48]:
expanded_roster_df = roster_view_df.loc[roster_view_df["Fantasy Team"]=="TEAM RAHEEM"]
expanded_roster_df = expanded_roster_df.drop("Fantasy Team", axis=1)

expanded_roster_df.head()

,Player,POS,TM,Points,Projected Points,Still Alive,Reg Szn PPG,Exp Games Remaining,WC PTS,DIV PTS,CONF PTS,SB PTS
0,ALLEN,QB,BUF,0,53.25,1,23.0,2.32,0,0,0,0
1,JACKSON,QB,BAL,0,46.81,1,23.1,2.03,0,0,0,0
2,GIBBS,RB,DET,0,36.55,1,18.0,2.03,0,0,0,0
3,JACOBS,RB,GB,0,23.85,1,16.0,1.49,0,0,0,0
4,BARKLEY,RB,PHI,0,44.92,1,20.0,2.25,0,0,0,0
